In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 2: Environment Detection + Repo Clone
# ═══════════════════════════════════════════════════════════════

import os, sys, platform, subprocess, warnings
warnings.filterwarnings('ignore')

# CRITICAL: set BEFORE any torch import to prevent OOM fragmentation
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

REPO_URL = 'https://github.com/RyoOtani/DS4SmallestAIprjct.git'
REPO_DIR = 'DS4SmallestAIprjct'

def _run_cmd(cmd):
    try:
        result = subprocess.run(cmd, shell=True, capture_output=True,
                                text=True, timeout=30)
        return [l for l in result.stdout.strip().split('\n') if l]
    except:
        return []

def _ensure_repo():
    if os.path.exists('src/main.c') and os.path.exists('Makefile'):
        return os.getcwd()
    if os.path.exists(f'{REPO_DIR}/src/main.c'):
        os.chdir(REPO_DIR)
        print(f"📂 Changed to repo: {os.getcwd()}")
        return os.getcwd()
    print(f"📥 Cloning repository: {REPO_URL}")
    _run_cmd(f'git clone --depth 1 {REPO_URL}')
    if os.path.exists(f'{REPO_DIR}/src/main.c'):
        os.chdir(REPO_DIR)
        print(f"✅ Cloned! Changed to: {os.getcwd()}")
        return os.getcwd()
    else:
        print(f"⚠️  Clone failed. Working in: {os.getcwd()}")
        return os.getcwd()

ENV = {}
ENV['python'] = sys.version
ENV['platform'] = platform.platform()

ENV['is_colab'] = False
try:
    import google.colab
    ENV['is_colab'] = True
except ImportError:
    pass

if ENV['is_colab']:
    print("✅ Environment: Google Colab")
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print("   Google Drive mounted at /content/drive")
    except Exception as e:
        print(f"   ⚠️ Drive mount skipped ({type(e).__name__})")
    gpu_lines = _run_cmd('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null')
    ENV['gpu'] = gpu_lines[0] if gpu_lines else "Unknown"
    print(f"   GPU: {ENV['gpu']}")
    _ensure_repo()
else:
    if os.environ.get('RUNPOD_POD_ID'):
        ENV['is_runpod'] = True
        print("✅ Environment: RunPod")
    elif os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
        ENV['is_kaggle'] = True
        print("✅ Environment: Kaggle")
    else:
        ENV['is_runpod'] = False
        ENV['is_kaggle'] = False
        print("✅ Environment: Local / Other")
    _ensure_repo()

import torch
ENV['gpu_count'] = torch.cuda.device_count() if torch.cuda.is_available() else 0
ENV['cuda_available'] = torch.cuda.is_available()
ENV['device'] = 'cuda' if torch.cuda.is_available() else 'cpu'
ENV['device_name'] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'

print(f"   Device:  {ENV['device_name']} x{max(ENV['gpu_count'], 1)}")
print(f"   CUDA:    {ENV['cuda_available']}")
print(f"   PyTorch: {torch.__version__}")
print(f"   CWD:     {os.getcwd()}")

if ENV['gpu_count'] >= 8:
    ENV['recommended_model'] = 'small'
    ENV['recommended_steps'] = 200000
elif ENV['gpu_count'] >= 4:
    ENV['recommended_model'] = 'nano'
    ENV['recommended_steps'] = 100000
elif ENV['gpu_count'] >= 1:
    ENV['recommended_model'] = 'nano'
    ENV['recommended_steps'] = 10000
else:
    ENV['recommended_model'] = 'nano'
    ENV['recommended_steps'] = 1000
    print("⚠️  No GPU detected! Training will be very slow (CPU only).")

print(f"\n💡 Recommended: --config {ENV['recommended_model']} for {ENV['recommended_steps']} steps")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 3: Install Dependencies
# ═══════════════════════════════════════════════════════════════

import subprocess, sys

def pip_install(*packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + list(packages))

# Core training deps
pip_install('torch>=2.4.0', 'transformers>=4.45.0', 'accelerate>=0.33.0',
            'datasets>=3.0.0', 'tokenizers>=0.20.0', 'wandb',
            'tqdm', 'numpy', 'safetensors', 'huggingface_hub')

# Install tinyllm package (editable)
if os.path.exists('setup.py'):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.'])
elif os.path.exists('requirements.txt'):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'])

# Build C runtime (optional, for inference after training)
if not ENV['is_colab']:
    try:
        subprocess.run(['make', '-C', '.'], capture_output=True, check=False)
        print("✅ C runtime built: ./tinyllm")
    except:
        print("⚠️  C runtime build skipped (not needed for training)")

# Verify imports
import torch
import transformers
import datasets
import accelerate
print(f"\n✅ torch={torch.__version__} transformers={transformers.__version__}")
print(f"✅ datasets={datasets.__version__} accelerate={accelerate.__version__}")

## 🏷️ Step 1: Tokenizer (offline-ready!)

TinyLLM bundles its own **production-grade ByteLevel BPE tokenizer**. **No internet needed.**

| Property | Value |
|----------|-------|
| Type | ByteLevel BPE (GPT-2 style) + NFKC normalization |
| Vocab size | **32,000** (21 specials + 256 bytes + 31,723 BPE merges) |
| Special tokens | `<s>`, `</s>`, `<pad>`, `<unk>`, FIM (`<fim_prefix>`, `<fim_suffix>`, `<fim_middle>`, `<fim_hole>`, `<fim_pad>`), Tool (`<tool_call>`, `<tool_response>`), Chat (`<|system|>`, `<|user|>`, `<|assistant|>`) |
| 🌍 Languages | 🇬🇧 English (0.40x) · 🇯🇵 Japanese (1.28x) · 🐍 Python (0.53x) · ⚡ C/JS/TS/Rust/Go/SQL |
| Python (bundled) | `tokenizer/` (~3.3 MB) — `AutoTokenizer.from_pretrained('tokenizer')` |
| C runtime | `tokenizer.tokbin` (950 KB) — `tl_tokenizer_load("tokenizer.tokbin")` |
| 🚀 Emoji | Full Unicode support (🚀🌍✨) |
| Rebuild | `python create_tokenizer.py` (any time, target any vocab size) |
| 🔍 Verify | `python test_3way_tokenizer.py` — Python↔C↔TOKBIN 完全一致確認 |

> **💡 日本語 + コード両対応**: ByteLevel BPE + NFKC 正規化により、日本語テキストもコードも高効率でトークン化。
> 英語 0.40x、Python 0.53x、日本語 1.28x の圧縮率。C ランタイムとは `.tokbin` バイナリで完全一致保証。
> 必要に応じて `python create_tokenizer.py --vocab 65536` でさらに大語彙も生成可能です。

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 9: Load Tokenizer (offline-first!)
# ═══════════════════════════════════════════════════════════════
#
# TinyLLM v7 トークナイザー (2026-07-25):
#   - 32,000 vocab · ByteLevel BPE · NFKC normalization
#   - 🇯🇵 日本語 · 🇬🇧 英語 · 🐍🦀⚡ コード すべてネイティブ対応
#   - FIM + Tool + Chat 特殊トークン完備
#
# Priority:
#   1. tokenizer/ (bundled) — 32K vocab, zero download, 🇯🇵 built-in
#   2. tinyllm-tokenizer/ (saved from previous run)
#   3. tokyotech-llm/Swallow-7b-v0.1 — 🇯🇵 fallback (Llama 2 based)
#   4. Qwen/Qwen2.5-1.5B (final fallback)

from transformers import AutoTokenizer
import os

TOKENIZER_OUT = 'tinyllm-tokenizer'

# ── Priority 1: Bundled 32K tokenizer ────────────────────────
if os.path.exists('tokenizer/tokenizer.json') and os.path.exists('tokenizer/tokenizer_config.json'):
    print("📥 Priority 1: bundled TinyLLM v7 tokenizer (32K vocab, offline)")
    tokenizer = AutoTokenizer.from_pretrained('tokenizer', use_fast=True)

# ── Priority 2: Saved from previous run ───────────────────────
elif os.path.exists(TOKENIZER_OUT):
    print(f"📥 Priority 2: saved tokenizer ({TOKENIZER_OUT}/)")
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_OUT, use_fast=True)

# ── Priority 3: Swallow (日本語フォールバック) ──────────────
elif True:  # ← switch to 'elif False:' to skip Swallow
    TOKENIZER_ID = "tokyotech-llm/Swallow-7b-v0.1"
    print(f"📥 Priority 3: Swallow-7B tokenizer 🇯🇵 (Japanese-optimized fallback)")
    print(f"   Source: {TOKENIZER_ID}")
    print(f"   Note: Swallow is based on Llama 2, has excellent Japanese tokenization")
    try:
        tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID, use_fast=True)
        tokenizer.save_pretrained(TOKENIZER_OUT)
    except Exception as e:
        print(f"   ⚠️ Swallow unavailable ({type(e).__name__}), trying Qwen fallback...")
        TOKENIZER_ID = "Qwen/Qwen2.5-1.5B"
        tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID, trust_remote_code=True, use_fast=True)
        tokenizer.save_pretrained(TOKENIZER_OUT)

# ── Priority 4: Qwen fallback ─────────────────────────────────
else:
    TOKENIZER_ID = "Qwen/Qwen2.5-1.5B"
    print(f"📥 Priority 4: Qwen tokenizer (65k vocab final fallback)")
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID, trust_remote_code=True, use_fast=True)
    tokenizer.save_pretrained(TOKENIZER_OUT)

# Add TinyLLM special tokens (already in v7 bundled tokenizer, but safe to re-add)
tokenizer.add_special_tokens({
    'additional_special_tokens': [
        '<fim_prefix>', '<fim_suffix>', '<fim_middle>',
        '<pad>', '<tool_call>', '</tool_call>',
        '<scratchpad>', '</scratchpad>',
    ]
})
if tokenizer.pad_token is None: tokenizer.pad_token = '<pad>'
if tokenizer.bos_token is None: tokenizer.bos_token = '<s>'
if tokenizer.eos_token is None: tokenizer.eos_token = '</s>'

print(f"✅ Tokenizer ready! vocab={len(tokenizer)}")
test = "def hello_world():\n    print('Hello, TinyLLM!')\n"
print(f"📝 '{test.strip()}' → {len(tokenizer.encode(test))} tokens")

## 📦 Step 2: Data Preparation — CodeParrot

> **⚠️ 先に Step 2 (Tokenizer) を実行してからこのセルを実行してください。**

認証不要の [CodeParrot](https://huggingface.co/datasets/codeparrot/codeparrot-clean) から
Python コードをダウンロードし、トークナイズして学習用バイナリに変換します。

| 設定 | 値 |
|------|-----|
| データセット | CodeParrot (Pythonコード) |
| 認証 | 不要 🆓 |
| 変換後 | `data/train.bin` + `data/val.bin` |


### 🔄 Data Source Options

| Source | Auth | Content |
|--------|------|--------|
| **CodeParrot** (default) | 不要 | Pythonコード |
| **The Stack v2** | 必要 | 高品質コード |
| **WikiText** (fallback) | 不要 | 英語テキスト |


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 7: Download & Tokenize Real Data
# ═══════════════════════════════════════════════════════════════
# 🔐 The Stack v2 needs auth: https://huggingface.co/datasets/bigcode/the-stack-v2-dedup
#
# 📌 Kaggle Secrets の設定方法:
#    1. Kaggle → Settings → Secrets → "Add Secret"
#    2. Label: HF_TOKEN, Value: hf_xxxxxxxxxxxxxxxxxxxx
#    3. このセルが自動的に HF_TOKEN を読み取ります

DATA_SOURCE = 'codeparrot'  # 'codeparrot' (free) or 'the_stack' (needs auth)

# ── Hugging Face login (for gated datasets) ────────────
import os
# Try Kaggle Secrets first, then environment variable
hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
except (ImportError, Exception):
    pass
if not hf_token:
    hf_token = os.environ.get('HF_TOKEN')
if hf_token:
    from huggingface_hub import login
    login(token=hf_token)
    print("✅ Logged in to Hugging Face via HF_TOKEN secret")
elif DATA_SOURCE == 'the_stack':
    print("⚠️  HF_TOKEN not set. The Stack v2 needs authentication.")
    print("   Add HF_TOKEN to Kaggle Secrets (Settings → Secrets)")
    print("   Or uncomment: from huggingface_hub import notebook_login; notebook_login()")

from datasets import load_dataset
import numpy as np

if DATA_SOURCE == 'the_stack':
    print("🔐 Downloading The Stack v2 (Python subset)...")
    ds = load_dataset("bigcode/the-stack-v2-dedup", data_dir="data/Python",
                      split="train", streaming=True, token=True).take(10000)
    print("✅ Loaded The Stack v2")
else:
    print("📦 Downloading CodeParrot (free, no auth needed)...")
    try:
        ds = load_dataset("codeparrot/codeparrot-clean", split="train", streaming=True).take(10000)
        print("✅ Loaded CodeParrot")
    except Exception as e:
        print(f"⚠️  CodeParrot failed: {e}")
        print("📦 Falling back to WikiText...")
        ds = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="train", streaming=True).take(10000)
        print("✅ Loaded WikiText")

# ── Tokenize ────────────────────────────────────────────
print(f"🔧 Tokenizing with vocab_size={len(tokenizer)}...")
all_tokens = []
for i, sample in enumerate(ds):
    text = sample.get("content") or sample.get("text") or sample.get("code") or ""
    if not text or len(text) < 10:
        continue
    ids = tokenizer.encode(text)
    all_tokens.extend(ids)
    if (i + 1) % 1000 == 0:
        print(f"   ... {i+1} samples, {len(all_tokens):,} tokens")
    if len(all_tokens) >= 5_000_000:
        break

tokens = np.array(all_tokens, dtype=np.int32)
train_split = int(len(tokens) * 0.9)
os.makedirs("data", exist_ok=True)
tokens[:train_split].tofile(os.path.join("data", "train.bin"))
tokens[train_split:].tofile(os.path.join("data", "val.bin"))

print(f"✅ Real data ready: train={train_split:,}, val={len(tokens)-train_split:,} tokens")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 5: Data Check
# ═══════════════════════════════════════════════════════════════
# Verifies that real data exists. If not, run Cell 7 first.

import os, numpy as np

if 'tokenizer' not in dir():
    print("⚠️  Tokenizer not loaded yet. Run Cell 9 first, then come back.")
    VOCAB_SIZE = 32000  # TinyLLM v7 bundled tokenizer
    SEQ_LEN = 1024
else:
    VOCAB_SIZE = len(tokenizer)
    SEQ_LEN = 1024

    if os.path.exists("data/train.bin"):
        data = np.memmap("data/train.bin", dtype=np.int32, mode="r")
        print(f"✅ Real data found: {len(data):,} tokens")
        print(f"   vocab={VOCAB_SIZE}, seq_len={SEQ_LEN}")
    else:
        print("📥 No real data yet. Run Cell 7 first.")


## 🧠 Step 3: Create Model — TinyLLM-nano (1.5B)

We create the **nano** model (1.5B active parameters, 1.5B total).  
This fits comfortably in a **T4 16GB** GPU with gradient checkpointing + mixed precision.

| Config | nano | small | medium |
|--------|------|-------|--------|
| Hidden dim | 1,024 | 2,048 | 4,096 |
| Layers | 24 | 32 | 48 |
| Attention heads | 16 | 32 | 48 |
| KV heads (GQA) | 4 | 8 | 8 |
| KV latent dim (MLA) | 256 | 512 | 1,024 |
| MoE experts | 32 | 64 | 128 |
| Active experts | 4 | 6 | 8 |
| Expert inter dim | 512 | 1,024 | 2,048 |
| **Total params** | **1.5B** | **14.8B** | **109B** |
| **Active params** | **1.5B** | **3.0B** | **6.5B** |
| GPU memory (BF16) | ~8 GB | ~24 GB | ~80 GB |

### 🔄 Resume from Checkpoint（チェックポイントから再開）

> **Kaggleなどで学習を中断→再開する場合の手順:**

1. **チェックポイントを保存**: 学習中に自動保存される `checkpoints/step_XXXX/model.pt` をKaggleのOutputからダウンロード
2. **次のセッションにアップロード**: Kaggleの「+ Add Data」→ 新しいDatasetとしてアップロード
3. **Cell 11 の `RESUME_FROM` にパスを設定**: 例: `/kaggle/input/my-checkpoint/model.pt`
4. **Cell 11 → 13 → 14 を順に実行**: 途中から学習が再開されます！

| 復元されるもの | 説明 |
|-------------|------|
| ✅ モデル重み | `model_state_dict` |
| ✅ optimizer状態 | `optimizer_state_dict` (momemtum等) |
| ✅ 学習率スケジューラ | 再開stepまで進める |
| ✅ step数 | `global_step` から継続 |

> **💡 ヒント**: The Stack v2 など実データで学習する場合、最初のセッションで数千step学習 → チェックポイント保存 → 次のセッションで継続、というサイクルを繰り返せば、無料GPUでも大規模学習が可能です。

In [ ]:
# 🔄 継続学習の設定（チェックポイントから再開する場合）
#    Kaggle: "/kaggle/input/データセット名/model.pt"
#    Colab:  "/content/drive/MyDrive/checkpoints/step_5000/model.pt"
#    Local:  "checkpoints/step_5000/model.pt"
#    新規学習の場合は None のまま
#    ⚠️ デフォルトではモデル重複のみ保存（~6GB）。optimizer状態（+12GB）は
#       必要な場合のみ Cell 13 の save_optimizer=True にしてください。
RESUME_FROM = None  # ← ここをチェックポイントのパスに変更！

## ⚡ Step 4: Training

Running 1,000 steps of pretraining with:
- **Mixed precision** (BF16/FP16) for memory efficiency
- **Gradient accumulation** (effective batch size = 32)
- **Cosine LR schedule** with linear warmup
- **WandB logging** (optional, set `USE_WANDB=True`)

> **Expected time**: ~15 min on T4, ~5 min on A100, ~2 min on H100

In [ ]:
# ── Training hyperparameters ─────────────────────────────────
TRAIN_CONFIG = {
    'max_steps': 1000,              # TOTAL steps (increase for real training: 100k+)
    'batch_size': 2,                # Per GPU
    'grad_accum': 8,                # Effective batch = 2 * 8 = 16
    'learning_rate': 3e-4,
    'warmup_steps': 100,
    'max_lr': 3e-4,
    'min_lr': 3e-5,
    'weight_decay': 0.1,
    'grad_clip': 1.0,
    'log_interval': 10,
    'save_interval': 500,
    'use_wandb': False,             # Set True for wandb logging
    'data_source': 'real',          # 'real' or 'dummy' (for quick test)
    'resume_optimizer': True,       # Resume optimizer state from checkpoint
    'save_optimizer': False,        # ⚠️ Save optimizer in checkpoint (+12GB!) — KaggleはFalse推奨
}

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 14: Training Loop — Run this to start/resume training!
# ═══════════════════════════════════════════════════════════════
#
# 🔄 チェックポイントから再開する場合:
#    - global_step は resumed_step から開始
#    - プログレスバーは残りステップ分だけ表示
#    - optimizer/scheduler は Cell 13 で復元済み
#
# 💾 ディスク容量節約のため、デフォルトではモデル重みのみ保存。
#    optimizer状態も保存する場合は Cell 13 で save_optimizer = True に。

from tqdm.notebook import tqdm
import json, os

torch.cuda.empty_cache()

cap = torch.cuda.get_device_capability() if torch.cuda.is_available() else (0, 0)
USE_BF16 = cap >= (8, 0)
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f"ℹ️  GPU sm_{cap[0]}{cap[1]}, using {'BF16' if USE_BF16 else 'FP16'}")

print("=" * 60)
if resumed_step > 0:
    print(f"🔄 Resuming: TinyLLM-{MODEL_SIZE}, step {resumed_step:,} → {TRAIN_CONFIG['max_steps']:,}")
else:
    print(f"🚀 Training: TinyLLM-{MODEL_SIZE}, {TRAIN_CONFIG['max_steps']} steps")
mem_used = torch.cuda.memory_allocated() / 1e9
if torch.cuda.is_available():
    mem_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"   GPU: {mem_used:.1f} / {mem_total:.1f} GB ({100*mem_used/mem_total:.0f}%)")
if gpu_count > 1:
    print(f"   🚀 DataParallel: {gpu_count} GPUs → effective batch = {TRAIN_CONFIG['batch_size'] * gpu_count * TRAIN_CONFIG['grad_accum']}")
print("=" * 60)

model.train()
optimizer.zero_grad()
torch.cuda.empty_cache()

global_step = resumed_step  # 🔄 Start from checkpoint step
total_loss = 0.0
start_time = time.time()
data_iter = iter(train_loader)
remaining = TRAIN_CONFIG['max_steps'] - resumed_step
pbar = tqdm(total=remaining, desc='Training', initial=0)

while global_step < TRAIN_CONFIG['max_steps']:
    try:
        batch = next(data_iter)
    except StopIteration:
        data_iter = iter(train_loader)
        batch = next(data_iter)

    input_ids = batch['input_ids'].to(device)
    labels = batch['labels'].to(device)

    with torch.cuda.amp.autocast(dtype=AMP_DTYPE):
        outputs = model(input_ids=input_ids, labels=labels)
        loss = outputs['loss'].mean()  # DataParallel may return per-GPU losses

    loss = loss / TRAIN_CONFIG['grad_accum']
    if USE_BF16: loss.backward()
    else: scaler.scale(loss).backward()
    total_loss += loss.item()

    if (global_step + 1) % TRAIN_CONFIG['grad_accum'] == 0:
        if USE_BF16:
            torch.nn.utils.clip_grad_norm_(model.parameters(), TRAIN_CONFIG['grad_clip'])
            optimizer.step()
        else:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), TRAIN_CONFIG['grad_clip'])
            scaler.step(optimizer)
            scaler.update()
        scheduler.step()
        optimizer.zero_grad()

    global_step += 1
    pbar.update(1)

    if global_step % TRAIN_CONFIG['log_interval'] == 0:
        avg_loss = total_loss / TRAIN_CONFIG['log_interval']
        elapsed = time.time() - start_time
        steps_done = global_step - resumed_step
        tok_per_sec = steps_done * TRAIN_CONFIG['batch_size'] * SEQ_LEN / elapsed if elapsed > 0 else 0
        pbar.set_postfix({'loss': f'{avg_loss:.4f}', 'tok/s': f'{tok_per_sec:.0f}'})
        total_loss = 0.0

    if global_step % TRAIN_CONFIG['save_interval'] == 0:
        ckpt = f'checkpoints/step_{global_step}'
        os.makedirs(ckpt, exist_ok=True)
        ckpt_dict = {
            'model_state_dict': {k: v.cpu() for k, v in model.state_dict().items()},
            'config': cfg_dict,
            'step': global_step,
        }
        # 💾 Optimizer state = ~12GB extra. Only save if disk has room.
        if TRAIN_CONFIG.get('save_optimizer', False):
            ckpt_dict['optimizer_state_dict'] = optimizer.state_dict()
        ckpt_path = f'{ckpt}/model.pt'
        try:
            torch.save(ckpt_dict, ckpt_path)
            ckpt_mb = os.path.getsize(ckpt_path) / (1024**2)
            print(f"\n💾 Checkpoint: {ckpt} ({ckpt_mb:.0f} MB, step {global_step:,})")
        except RuntimeError as e:
            print(f"\n⚠️  Save failed (disk full?): {e}")
            print(f"   Continuing without saving this checkpoint...")
        torch.cuda.empty_cache()

pbar.close()
elapsed = time.time() - start_time
steps_done = global_step - resumed_step
tok_per_sec = steps_done * TRAIN_CONFIG['batch_size'] * SEQ_LEN / elapsed if elapsed > 0 else 0
print(f"\n✅ Complete! {elapsed:.0f}s, {tok_per_sec:.0f} tok/s")
print(f"   Total steps: {resumed_step:,} → {global_step:,} ({steps_done:,} new)")
print(f"   Peak GPU: {torch.cuda.max_memory_allocated()/1e9:.1f} GB")

# ── Final save ─────────────────────────────────────────────
final_dir = 'checkpoints/final'
os.makedirs(final_dir, exist_ok=True)
final_dict = {
    'model_state_dict': {k: v.cpu() for k, v in model.state_dict().items()},
    'config': cfg_dict,
    'step': global_step,
}
if TRAIN_CONFIG.get('save_optimizer', False):
    final_dict['optimizer_state_dict'] = optimizer.state_dict()
torch.save(final_dict, f'{final_dir}/model.pt')
tokenizer.save_pretrained(final_dir)
with open(f'{final_dir}/config.json', 'w') as f:
    json.dump(cfg_dict, f, indent=2)
final_mb = os.path.getsize(f'{final_dir}/model.pt') / (1024**2)
print(f"💾 Final model saved to {final_dir}/ ({final_mb:.0f} MB, step {global_step:,})")

## 📦 Step 5: Export — HuggingFace + GGUF + TOKBIN

Export trained model in three formats:

| Format | Command | For |
|--------|---------|-----|
| **HuggingFace** | (this cell) | Python inference, sharing |
| **GGUF FP16** | `python export_gguf.py` | C runtime (1.2 GB) |
| **GGUF Q4_K_M** | `python export_gguf.py --q4_k` | C runtime (350 MB, 4-bit) |
| **TOKBIN** | `python export_tokenizer.py` | C tokenizer (950 KB) |

The C runtime uses a **single unified binary** — GGUF model + TOKBIN tokenizer.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 16: Export Model (HuggingFace + TOKBIN + Q4 GGUF)
# ═══════════════════════════════════════════════════════════════
# Saves in 3 formats:
#   1. HF-compatible (model.pt + config.json + tokenizer/)
#   2. TOKBIN (C runtime tokenizer: tokenizer.tokbin)
#   3. GGUF Q4 (C runtime model: tinyllm-nano-q4.gguf)
#
# 🔧 Works on Colab/Kaggle/Local — auto-finds repo directory.

EXPORT_Q4 = True           # Set False to skip Q4 GGUF export
EXPORT_TOKBIN = True        # Set False to skip TOKBIN export

save_dir = f'checkpoints/final'
os.makedirs(save_dir, exist_ok=True)

# ── Find repo root (where export scripts live) ─────────────
REPO_ROOT = os.getcwd()
for candidate in [REPO_DIR, '.', 'DS4SmallestAIprjct',
                  '/kaggle/working/DS4SmallestAIprjct']:
    if os.path.exists(f'{candidate}/export_tokenizer.py'):
        REPO_ROOT = os.path.abspath(candidate)
        break
print(f"📂 Repo root: {REPO_ROOT}")

print("💾 Saving model...")

# ── 1. HuggingFace format ─────────────────────────────────
torch.save({'model_state_dict': model.state_dict(),
            'config': cfg_dict,
            'model_size': MODEL_SIZE,
            'vocab_size': len(tokenizer)},
           f'{save_dir}/model.pt')

import json
with open(f'{save_dir}/config.json', 'w') as f:
    json.dump(cfg_dict, f, indent=2)

tokenizer.save_pretrained(save_dir)

size_mb = os.path.getsize(f'{save_dir}/model.pt') / (1024 * 1024)
print(f"   ✅ HF format: {save_dir}/ ({size_mb:.0f} MB)")

# ── 2. TOKBIN (C runtime tokenizer) ───────────────────────
if EXPORT_TOKBIN:
    export_tok = os.path.join(REPO_ROOT, 'export_tokenizer.py')
    if os.path.exists(export_tok):
        print("🔧 Exporting TOKBIN for C runtime...")
        subprocess.run([sys.executable, export_tok,
                        '--input', f'{save_dir}/tokenizer.json',
                        '--output', 'tokenizer.tokbin'], check=True)
        tokbin_kb = os.path.getsize('tokenizer.tokbin') / 1024
        print(f"   ✅ TOKBIN: tokenizer.tokbin ({tokbin_kb:.0f} KB)")
    else:
        print(f"   ⚠️  export_tokenizer.py not found at {REPO_ROOT}")
        print(f"   💡 Clone repo or run: python export_tokenizer.py manually")

# ── 3. Q4 GGUF (C runtime model, 4-bit quantized) ─────────
if EXPORT_Q4:
    export_gguf = os.path.join(REPO_ROOT, 'export_gguf.py')
    if os.path.exists(export_gguf):
        print("🔧 Exporting Q4_K_M GGUF (~4.5 bpw)...")
        subprocess.run([sys.executable, export_gguf,
                        '--q4_k', '--output', 'tinyllm-nano-q4.gguf'], check=True)
        gguf_mb = os.path.getsize('tinyllm-nano-q4.gguf') / (1024 * 1024)
        print(f"   ✅ Q4 GGUF: tinyllm-nano-q4.gguf ({gguf_mb:.0f} MB)")
    else:
        print(f"   ⚠️  export_gguf.py not found, skipping Q4 GGUF")

print(f"\n✅ All exports complete!")
print(f"   📁 {save_dir}/          — HuggingFace (Python)")
if EXPORT_TOKBIN:
    print(f"   📁 tokenizer.tokbin     — C tokenizer (950 KB)")
if EXPORT_Q4:
    print(f"   📁 tinyllm-nano-q4.gguf — C runtime (Q4, ~350 MB)")
print(f"\n💡 C runtime: ./tinyllm run tinyllm-nano-q4.gguf")
print(f"💡 Verify:    python test_3way_tokenizer.py")

# 🚀 TinyLLM Training Benchmark — 1-Click Pretraining

> **Open this notebook in Colab or RunPod → Run all cells → Get a trained TinyLLM model in hours**

| Platform | Badge | Notes |
|----------|-------|-------|
| **Google Colab** | [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RyoOtani/DS4SmallestAIprjct/blob/main/TINYLLM_TRAIN_BENCHMARK.ipynb) | Free T4 GPU, nano model |
| **RunPod** | [![RunPod](https://img.shields.io/badge/RunPod-1--click-blue)](https://runpod.io/console/deploy?template=) | A100/H100, up to medium model |
| **Kaggle** | [![Kaggle](https://img.shields.io/badge/Kaggle-Notebook-blue)](https://kaggle.com/) | 2× T4, nano/small model |

---

## 📋 What You'll Get

| Step | What happens | Time |
|------|-------------|------|
| 1 | Auto-detect environment (Colab/RunPod/Local) | 10s |
| 2 | Load bundled **32K ByteLevel BPE tokenizer** (🇯🇵+code) | 5s |
| 3 | Download CodeParrot + tokenize | 30s – 5min |
| 4 | Install dependencies | 2min |
| 5 | Create **TinyLLM-nano** model (1.5B params) | 30s |
| 6 | Train on GPU for 1,000 steps | 15–30min |
| 7 | Export: HF + TOKBIN + Q4 GGUF | 2min |
| 8 | (Optional) Upload to Hugging Face Hub | 2min |

---

## 🗺️ Roadmap

| Week | Feature | Status |
|------|---------|--------|
| ✅ Week 1 | 32K ByteLevel BPE tokenizer (🇯🇵日本語 + 🐍🦀⚡コード) + CodeParrot training | **Done** |
| ✅ Week 2 | Paged KV cache + Continuous batching (vLLM-style) | **Done** |
| ✅ Week 3 | 4-bit 量子化 (Q4_0/Q4_K_M) + 知識蒸留パイプライン (DeepSeek→TinyLLM) | **Done** |
| ✅ Fix | C tokenizer → 実TOKBINローダー (512ダミー廃止) + 3-way整合性テスト | **Done** |
| 🔜 Next | C ランタイム paged KV 対応 + GGUF Q4 推論最適化 | 次回 |

---

## 🎯 Target Audience

**GPU-rich engineers who hate environment setup.**  
If you have an idle A100/H100/RTX 4090, this notebook will make it actually *do* something useful overnight.

---

## 日本語概要

このノートブックを開いてセルを上から実行するだけで、TinyLLM モデルの事前学習をテストできます。

- **トークナイザー**: 32K 語彙 ByteLevel BPE（日本語・英語・コード ネイティブ対応）
- **C ランタイム**: TOKBIN (950KB) + Q4 GGUF (~350MB) で完全一致
- **検証**: `test_3way_tokenizer.py` で Python↔C↔TOKBIN 完全一致保証
- **デフォルト**: CodeParrot（認証不要）で即時動作確認
- **本番モード**: The Stack v2 (Hugging Face) で本格的なコード事前学習

学習が完了したら、Hugging Face のコミュニティリポジトリにアップロードして世界と共有しましょう！

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 18: 🔄 Auto-Upload Checkpoint to HuggingFace
# ═══════════════════════════════════════════════════════════════
# Uploads trained model to HF. Next session auto-resumes from it.
# Set HF_TOKEN in Kaggle Secrets → Settings → Secrets.

import os

# ── Login ──────────────────────────────────────────
# Try Kaggle Secrets first, then environment variable
hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
except (ImportError, Exception):
    pass
if not hf_token:
    hf_token = os.environ.get('HF_TOKEN')
if hf_token:
    from huggingface_hub import login
    login(token=hf_token)
    print("✅ Logged in to HuggingFace")
else:
    print("⚠️  HF_TOKEN not set. Skipping upload.")
    print("   Add HF_TOKEN in Kaggle Secrets to enable auto-upload.")

# ── Upload checkpoint ──────────────────────────────
if hf_token and os.path.exists('checkpoints/final/model.pt'):
    from huggingface_hub import HfApi
    api = HfApi()
    ckpt = os.path.getsize("checkpoints/final/model.pt") / 1e9
    print(f"📤 Uploading checkpoint ({ckpt:.2f} GB)...")
    api.upload_file(
        path_or_fileobj="checkpoints/final/model.pt",
        path_in_repo="tinyllm-nano/model.pt",
        repo_id="Ryo3desu/tinyllm-models",
        repo_type="model",
    )
    print("✅ Uploaded! Next Kaggle session will auto-resume.")
elif hf_token:
    print("⚠️  checkpoints/final/model.pt not found. Did training complete?")


## 🌍 Step 6: Share on Hugging Face 🤗

**You trained a model. Now show the world!**

Upload your trained weights to the community repository:

> **🤗 [https://huggingface.co/Ryo3desu/tinyllm-models](https://huggingface.co/Ryo3desu/tinyllm-models)**

This is the **official collection** for community-trained TinyLLM models.  
Every contributor gets credited in the model card.

**What to upload:**
- `checkpoints/final/` — HuggingFace format (model.pt + tokenizer/)
- `tinyllm-nano-q4.gguf` — Q4 quantized for C runtime (~350 MB)
- `tokenizer.tokbin` — C runtime tokenizer (950 KB)

**Why share?**
- Your name in the contributor hall of fame
- Other engineers build on your work
- You get feedback and improvements from the community

## 📊 Benchmark Results

Training speed benchmark across different GPUs.  
Update this cell after your run to help the community!

| GPU | Model | Tokens/sec | Loss (1k steps) | Time |
|-----|-------|-----------|-----------------|------|
| T4 (Colab) | nano | ~2,000 | ~10.5 | ~15 min |
| RTX 4090 | nano | ~8,000 | ~10.5 | ~4 min |
| A100 80GB | nano | ~20,000 | ~10.5 | ~2 min |
| A100 80GB | small | ~8,000 | ~11.2 | ~20 min |
| H100 80GB | small | ~15,000 | ~11.2 | ~10 min |

> **Note**: Loss values are for dummy data — real data will have lower loss.

---
*Generated by TinyLLM Training Benchmark — [Report issues here](https://github.com/RyoOtani/DS4SmallestAIprjct/issues)*